# Practical Assignment 1: Word Embeddings (Word2Vec)

**Course:** Deep Learning for Natural Language Processing (NLP)  
**Semester:** 2026/1  
**Institution:** Universidade Federal de Minas Gerais (UFMG)  
**Author:** Gustavo Freitas Cunha  
**Subject:** Systematic Hyperparameter Exploration and Intrinsic Analogy Evaluation in Word2Vec  

---

### 🤖 AI Disclosure & Academic Integrity Statement

> **Institutional Compliance & Transparency Notice:**  
> In compliance with academic integrity standards and the institutional guidelines on the ethical and responsible use of Artificial Intelligence established by the **Universidade Federal de Minas Gerais (UFMG)** ([UFMG Diretrizes Institucionais sobre IA](https://www.ufmg.br/ia/)), this work explicitly declares the scope and boundaries of AI tool assistance.  
> 
> **Scope of AI Assistance:**  
> An agentic AI coding assistant (Google Antigravity) was utilized strictly as an auxiliary tool for repository scaffolding, notebook formatting structure, dependency boilerplate, and routine data ingestion workflows.  
> 
> **Student Authorship & Responsibility:**  
> **All conceptual formulations, scientific hypotheses, hyperparameter selection strategies, experimental designs, critical analyses, empirical evaluations, and written interpretations were directed, developed, and authored by the student**.  
> 
> **Auditing & Validation:**  
> Every line of code, documentation segment, and pipeline step suggested or formatted with AI assistance was **rigorously reviewed, validated, and empirically tested** by the student to ensure scientific validity, factual accuracy, and full adherence to institutional and pedagogical standards.

# 1. Setup & Environment

This section initializes the runtime environment, installs required libraries, and imports scientific packages for modeling, vector linear algebra, and data manipulation. We configure global logging aesthetics and fix random seeds to ensure experimental reproducibility.

In [ ]:
#CELL 1.1 - INSTALL REQUIRED DEPENDENCIES
!pip install -q gensim numpy scipy matplotlib seaborn pandas

In [ ]:
#CELL 1.2 - IMPORT DEPENDENCIES AND CONFIGURE REPRODUCIBILITY ENVIRONMENT
import os
import sys
import time
import zipfile
import urllib.request
import logging
from collections import Counter
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any, Set

import numpy as np
import pandas as pd
import scipy.spatial.distance as scipy_dist
import matplotlib.pyplot as plt
import seaborn as sns

import gensim
from gensim.models import Word2Vec
from gensim.models.word2vec import Text8Corpus

#configure logging and visualization aesthetics
logging.basicConfig(
    format="%(asctime)s : %(levelname)s : %(message)s",
    level=logging.INFO
)
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 120

#fix deterministic seed
GLOBAL_RANDOM_SEED = 42
np.random.seed(GLOBAL_RANDOM_SEED)
print(f"Environment ready. Gensim version: {gensim.__version__} | Pandas version: {pd.__version__}")

# 2. Data Acquisition, Preprocessing & Corpus Preparation

In this section, we acquire, clean, and profile the two datasets required for training and evaluation:

1. **`text8` Corpus:** The first $10^8$ characters of cleaned Wikipedia text (~17 million tokens), lowercased and stripped of punctuation. We wrap this dataset using Gensim's `Text8Corpus` generator, which streams sentences iteratively from disk to prevent excessive memory allocation.
2. **`questions-words.txt` Dataset:** A benchmark of ~19,500 proportional analogy quadruplets spanning syntactic categories (e.g., comparative adjectives, verb inflections) and semantic categories (e.g., capitals-countries, currency-country).

### Preprocessing and Filtering Strategy
- **Case Normalization:** Because `text8` contains solely lowercased tokens, all analogy words are normalized to lowercase to maintain vocabulary alignment.
- **Deduplication:** Repeated analogy instances within identical categories are filtered out.
- **Frequency Profiling & Vocabulary Filtering:** Words occurring fewer than a minimum frequency threshold ($f < 5$) are identified as *hapax legomena* or rare tokens. We audit the analogy quadruplets against the corpus vocabulary to filter out instances containing out-of-vocabulary (OOV) tokens, ensuring that all subsequent mathematical evaluations operate on fully represented vector pairs.

In [ ]:
#CELL 2.1 - CONFIGURE DATA PATHS AND DOWNLOAD BENCHMARK DATASETS
#resolve data directory across local and cloud environments
if Path("data").exists():
    DATA_DIR = Path("data")
elif Path("../data").exists():
    DATA_DIR = Path("../data")
elif Path("TP1/data").exists():
    DATA_DIR = Path("TP1/data")
else:
    DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

TEXT8_EXTRACTED_PATH = DATA_DIR / "text8"
TEXT8_ZIP_PATH = DATA_DIR / "text8.zip"
QUESTIONS_FILE_PATH = DATA_DIR / "questions-words.txt"

TEXT8_URL = "http://mattmahoney.net/dc/text8.zip"
QUESTIONS_URL = "https://raw.githubusercontent.com/nicholas-leonard/word2vec/master/questions-words.txt"

#check existing files to prevent redundant downloads and duplicates
if TEXT8_EXTRACTED_PATH.exists():
    print(f"Verified existing text8 corpus at: {TEXT8_EXTRACTED_PATH.resolve()} (skipping download to prevent duplicates)")
else:
    print(f"Downloading text8 archive from {TEXT8_URL}...")
    urllib.request.urlretrieve(TEXT8_URL, TEXT8_ZIP_PATH)
    print("Extracting text8 corpus...")
    with zipfile.ZipFile(TEXT8_ZIP_PATH, "r") as zf:
        zf.extractall(DATA_DIR)
    #remove temporary archive to avoid duplicates and save disk space
    if TEXT8_ZIP_PATH.exists():
        TEXT8_ZIP_PATH.unlink()
    print("Extraction complete and temporary archive removed.")

#download analogy benchmark only if missing
if QUESTIONS_FILE_PATH.exists():
    print(f"Verified existing analogy dataset at: {QUESTIONS_FILE_PATH.resolve()} (skipping download to prevent duplicates)")
else:
    print(f"Downloading questions-words.txt from {QUESTIONS_URL}...")
    urllib.request.urlretrieve(QUESTIONS_URL, QUESTIONS_FILE_PATH)
    print("Download complete.")

print(f"\nData verification summary:")
print(f"- text8 corpus: {TEXT8_EXTRACTED_PATH.stat().st_size / (1024 * 1024):.2f} MB")
print(f"- questions-words dataset: {QUESTIONS_FILE_PATH.stat().st_size / 1024:.2f} KB")

In [ ]:
#CELL 2.2 - PARSE AND CLEAN ANALOGY BENCHMARK INTO STRUCTURED DATAFRAME
#parse raw text format into structured records
raw_records: List[Dict[str, str]] = []
active_category: Optional[str] = None

with open(QUESTIONS_FILE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        if line.startswith(":"):
            active_category = line[1:].strip()
        else:
            tokens = line.lower().split()
            #verify quadruplet token integrity
            if len(tokens) == 4 and active_category is not None:
                section_type = "syntactic" if active_category.startswith("gram") else "semantic"
                raw_records.append({
                    "category": active_category,
                    "section_type": section_type,
                    "word_a": tokens[0],
                    "word_b": tokens[1],
                    "word_c": tokens[2],
                    "word_d": tokens[3]
                })

df_analogies_raw = pd.DataFrame(raw_records)

#filter out duplicate rows within the same category
initial_count = len(df_analogies_raw)
df_analogies = df_analogies_raw.drop_duplicates(
    subset=["category", "word_a", "word_b", "word_c", "word_d"]
).reset_index(drop=True)
duplicates_removed = initial_count - len(df_analogies)

print(f"Analogy parsing and cleaning completed:")
print(f"- Total records parsed: {initial_count}")
print(f"- Duplicates removed: {duplicates_removed}")
print(f"- Clean unique quadruplets: {len(df_analogies)}")
print(f"- Section distribution:\n{df_analogies['section_type'].value_counts()}")
display(df_analogies.head(5))

In [ ]:
#CELL 2.3 - INITIALIZE STREAMING CORPUS AND CONSTRUCT VOCABULARY FREQUENCY PROFILE
#stream corpus from disk without full in-memory loading
corpus = Text8Corpus(str(TEXT8_EXTRACTED_PATH))

#profile word frequency distribution across corpus
vocab_counter: Counter = Counter()
total_corpus_tokens = 0

with open(TEXT8_EXTRACTED_PATH, "r", encoding="utf-8") as f:
    for line in f:
        tokens = line.split()
        total_corpus_tokens += len(tokens)
        vocab_counter.update(tokens)

unique_raw_words = len(vocab_counter)
#filter vocabulary threshold matching word2vec default min_count
MIN_WORD_COUNT = 5
vocab_filtered: Set[str] = {w for w, c in vocab_counter.items() if c >= MIN_WORD_COUNT}

print(f"Corpus frequency profiling completed:")
print(f"- Total running tokens: {total_corpus_tokens:,}")
print(f"- Total unique raw words: {unique_raw_words:,}")
print(f"- Filtered vocabulary (frequency >= {MIN_WORD_COUNT}): {len(vocab_filtered):,} words")
print(f"- Infrequent words pruned (< {MIN_WORD_COUNT}): {unique_raw_words - len(vocab_filtered):,} words")

In [ ]:
#CELL 2.4 - AUDIT VOCABULARY OVERLAP AND TAG IN-VOCABULARY ANALOGIES
#tag analogy quadruplets where all four words are present in filtered vocabulary
def verify_quadruplet_in_vocab(row: pd.Series, vocab: Set[str]) -> bool:
    return (
        row["word_a"] in vocab and
        row["word_b"] in vocab and
        row["word_c"] in vocab and
        row["word_d"] in vocab
    )

df_analogies["in_vocab"] = df_analogies.apply(
    lambda r: verify_quadruplet_in_vocab(r, vocab_filtered),
    axis=1
)

valid_analogies_count = df_analogies["in_vocab"].sum()
total_analogies = len(df_analogies)
coverage_pct = (valid_analogies_count / total_analogies) * 100

#filter dedicated dataframe for subsequent model evaluation
df_analogies_valid = df_analogies[df_analogies["in_vocab"]].copy().reset_index(drop=True)

print(f"Vocabulary audit on analogy benchmark (min_count={MIN_WORD_COUNT}):")
print(f"- Fully covered quadruplets (ready for evaluation): {valid_analogies_count:,} ({coverage_pct:.2f}%)")
print(f"- Quadruplets with OOV tokens (excluded from evaluation): {total_analogies - valid_analogies_count:,}")

#breakdown coverage by category
category_audit = df_analogies.groupby(["section_type", "category"])["in_vocab"].agg(
    total_quadruplets="count",
    covered_quadruplets="sum",
    coverage_ratio="mean"
).reset_index()
category_audit["coverage_pct"] = category_audit["coverage_ratio"] * 100

display(category_audit.sort_values(by="coverage_pct", ascending=False))